# 🫁 PulmoVision
## Explainable Chest X-ray Disease Classification using Transfer Learning

**Team:** Ishwar (23XV1M0518) · Mokshagna (23XV1M0542) · Mohit (23XV1M0528) · Varun (23XV1M0549)
**Academic Year:** 2024-25 | **Version:** 2.0

> ⚠️ **Disclaimer:** This is an **academic prototype** for educational purposes only.
> It is **NOT** validated for clinical use and must **not** be used to make medical decisions.

---

### 📋 Notebook Steps
| # | Section |
|---|---------|
| 1 | Install Dependencies & Mount Google Drive |
| 2 | Download Dataset to Google Drive |
| 3 | Import Libraries & Constants |
| 4 | Data Exploration |
| 5 | Prepare Dataset (Train / Val / Test split) |
| 6 | Build Data Generators (per-model preprocessing) |
| 7 | MLflow Experiment Tracking Setup |
| 8 | Model Architecture Builder |
| 9 | Training Function (2-Phase with safe fine-tuning) |
| 10–12 | Train EfficientNet-B3 / DenseNet-121 / ResNet-50 |
| 13 | Load Saved Models (2-model local variant) |
| 14 | Evaluate Individual Models on Test Set |
| 15 | Ensemble Soft Voting |
| 16 | Confusion Matrices |
| 17 | Grad-CAM++ Explainability Heatmaps |
| 18 | FastAPI Backend (Local) |

### 🔧 Fixes in v2.0
- Per-model preprocessing (no more double rescaling)
- Phase 2 backbone detection via layer names (not `model.layers[0]`)
- Conservative fine-tuning: 10% unfreeze, lr=5e-6, BatchNorm frozen
- Dataset persisted on Google Drive (no re-download each session)
- Class weights for imbalanced dataset
- Generator resets between training phases
- Phase 1 checkpoint protection
- Robust error handling and logging throughout
- Local FastAPI backend for deployment without ngrok

## ⚙️ Step 1 — Install Dependencies & Mount Google Drive

`CPU is fine for Step 1 and Step 2 (install + dataset download/extract).`
`Switch to T4 GPU before Step 3 and training.`

First run takes ~2-3 minutes for installs.

In [ ]:
# ── Install all required packages ─────────────────────────────────────────────
# Compatibility setup for modern Colab (Python 3.12 + TF 2.19)
# NOTE: No quiet flags — we want full output and progress bars for debugging

!pip install --progress-bar on "tf-keras"
!pip install --progress-bar on "tf-keras-vis>=0.8.0"
!pip install --progress-bar on "fastapi>=0.115.2"
!pip install --progress-bar on "uvicorn[standard]"
!pip install --progress-bar on "python-multipart"
!pip install --progress-bar on "mlflow"    # optional in practice; Step 7 already has fallback
!pip install --progress-bar on "nest-asyncio"
!pip install --progress-bar on "kaggle"
!pip install --progress-bar on "opencv-python-headless"
!pip install --progress-bar on "Pillow"
!pip install --progress-bar on "seaborn"
!pip install --progress-bar on "scikit-learn"    # pre-installed on Colab but keep explicit
!pip install --progress-bar on "scipy"           # explicit dependency
!pip install --progress-bar on "requests"        # used in health-check test cell
!pip install --progress-bar on "tqdm"

# ── Verify critical imports without side effects ─────────────────────────────
import os, importlib, sys
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')

required_specs = {
    'tensorflow':   'tensorflow',
    'fastapi':      'fastapi',
    'uvicorn':      'uvicorn',
    'tf_keras':     'tf-keras',
    'tf_keras_vis': 'tf-keras-vis',
    'PIL':          'Pillow',
    'cv2':          'opencv-python-headless',
    'multipart':    'python-multipart',
    'nest_asyncio': 'nest-asyncio',
    'sklearn':      'scikit-learn',
    'seaborn':      'seaborn',
    'scipy':        'scipy',
    'requests':     'requests',
    'tqdm':         'tqdm',
}

optional_specs = {
    'mlflow': 'mlflow',
    'kaggle': 'kaggle',  # avoid importing kaggle directly here (it can trigger auth code)
}

def find_missing(spec_map):
    missing = []
    for module_name, pip_name in spec_map.items():
        if importlib.util.find_spec(module_name) is None:
            missing.append(pip_name)
    return missing

missing_required = find_missing(required_specs)
if missing_required:
    raise RuntimeError(
        f"❌ Missing required packages: {missing_required}\n"
        f"   Re-run the install cell or run: pip install {' '.join(missing_required)}"
    )

missing_optional = find_missing(optional_specs)
if missing_optional:
    print(f"⚠️ Optional packages missing: {missing_optional}")
    print("   Notebook can continue, but related features may be limited.")

import tensorflow as tf
print("✅ Package verification complete!")
print(f"   Python      : {sys.version.split()[0]}")
print(f"   TensorFlow  : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"   GPU devices : {gpus}")
if not gpus:
    print("⚠️ No GPU detected. CPU is fine for install + dataset download/extract.")
    print("   Switch to T4 before Step 3 training flow.")

In [ ]:
from google.colab import drive
import os, logging

# ── Setup logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('PulmoVision')

# ── Mount Google Drive ─────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Project directory structure on Google Drive ────────────────────────────
PROJECT_DIR  = '/content/drive/MyDrive/Pulmoo'
MODELS_DIR   = os.path.join(PROJECT_DIR, 'models')
MLRUNS_DIR   = os.path.join(PROJECT_DIR, 'mlruns')
LOGS_DIR     = os.path.join(PROJECT_DIR, 'logs')
DATA_DIR     = os.path.join(PROJECT_DIR, 'dataset_raw')    # Raw Kaggle download
DATASET_DIR  = os.path.join(PROJECT_DIR, 'dataset_split')  # Train/val/test split

for d in [PROJECT_DIR, MODELS_DIR, MLRUNS_DIR, LOGS_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

log.info("Google Drive mounted")
log.info(f"Project root : {PROJECT_DIR}")
log.info(f"  models/      → saved model weights (.h5)")
log.info(f"  mlruns/      → MLflow experiment logs")
log.info(f"  logs/        → training curves & heatmaps")
log.info(f"  dataset_raw/ → raw Kaggle dataset (persistent)")
log.info(f"  dataset_split/ → train/val/test split (persistent)")

## 📥 Step 2 — Download Dataset to Google Drive

**Dataset:** COVID-19 Radiography Database (Tawsifur Rahman et al.) — ~1.8 GB, ~21,000 images
**Classes:** COVID-19 · Normal · Lung Opacity · Viral Pneumonia

The dataset is downloaded **to Google Drive** so it persists across Colab sessions.
Logic: Check Drive first → only download if not present.

### Before running this cell:
1. Go to [kaggle.com](https://www.kaggle.com) → **Settings → API → Create New Token**
2. This downloads a `kaggle.json` file to your computer
3. Upload it when the prompt appears below

> ⏭️ **Skip this step** if dataset is already on Drive.

In [ ]:
from google.colab import files
import zipfile
import shutil
from tqdm.auto import tqdm

# ── The expected final path after extraction ───────────────────────────────
RAW_DATASET_PATH = os.path.join(DATA_DIR, 'COVID-19_Radiography_Dataset')

def dataset_is_valid(path):
    '''Check if the dataset directory exists and has all 4 class folders with images.'''
    if not os.path.isdir(path):
        return False
    expected_classes = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
    for cls in expected_classes:
        cls_dir = os.path.join(path, cls, 'images')
        if not os.path.isdir(cls_dir):
            cls_dir = os.path.join(path, cls)
        if not os.path.isdir(cls_dir):
            log.warning(f"Missing class directory: {cls}")
            return False
        n_images = len([f for f in os.listdir(cls_dir)
                        if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if n_images < 100:
            log.warning(f"Class '{cls}' has only {n_images} images — seems incomplete")
            return False
    return True

# ── Check if dataset already exists on Drive ───────────────────────────────
if dataset_is_valid(RAW_DATASET_PATH):
    log.info(f"✅ Dataset already exists on Drive: {RAW_DATASET_PATH}")
    log.info("   Skipping download.")
else:
    log.info("Dataset not found on Drive. Starting download...")

    # Upload kaggle.json
    print("📁 Please upload your kaggle.json file:")
    try:
        uploaded = files.upload()
    except Exception as e:
        raise RuntimeError(
            f"❌ kaggle.json upload failed: {e}\n"
            "   Go to kaggle.com → Settings → API → Create New Token"
        )

    if not uploaded:
        raise RuntimeError(
            "❌ No file uploaded. Please upload your kaggle.json file."
        )

    uploaded_name = list(uploaded.keys())[0]
    if 'kaggle' not in uploaded_name.lower() or not uploaded_name.lower().endswith('.json'):
        raise RuntimeError(
            f"❌ Expected a kaggle.json file but got '{uploaded_name}'."
        )

    try:
        kaggle_dir = os.path.expanduser('~/.kaggle')
        os.makedirs(kaggle_dir, exist_ok=True)
        kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
        shutil.copy2(uploaded_name, kaggle_json_path)
        os.chmod(kaggle_json_path, 0o600)
        log.info("✅ kaggle.json configured")
    except Exception as e:
        raise RuntimeError(f"❌ Failed to configure kaggle.json: {e}")

    # Download to local Colab storage first (faster than direct Drive download)
    log.info("📥 Downloading COVID-19 Radiography Dataset (~1.8 GB) ...")
    !kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /content/

    # Find the downloaded zip
    zip_path = '/content/covid19-radiography-database.zip'
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f"❌ Download failed — zip not found at {zip_path}\n"
            "   Check your kaggle.json credentials and internet connection."
        )

    log.info(f"📦 Extracting to Google Drive ({DATA_DIR}) ...")
    log.info("   This can take several minutes — progress is shown below.")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.infolist()
        for member in tqdm(members, desc='Extracting dataset', unit='file'):
            zf.extract(member, DATA_DIR)

    # Clean up temp zip
    os.remove(zip_path)
    log.info("🗑️  Cleaned up temp zip file")

    # Validate
    if dataset_is_valid(RAW_DATASET_PATH):
        log.info("✅ Dataset downloaded and verified on Google Drive!")
    else:
        raise RuntimeError(
            f"❌ Dataset extraction seems incomplete at {RAW_DATASET_PATH}\n"
            "   Check the folder manually on Drive."
        )

# ── Print final dataset location ───────────────────────────────────────────────────────────
log.info(f"Dataset path: {RAW_DATASET_PATH}")

## 📚 Step 3 — Import Libraries & Set Constants

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import shutil
import warnings
import gc
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3, DenseNet121, ResNet50

# ✅ FIX: Import per-model preprocessing functions
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping,
                                        ReduceLROnPlateau)
from tensorflow.keras.optimizers import Adam

try:
    import mlflow
    import mlflow.keras
except Exception as e:
    mlflow = None
    log.warning(f"MLflow import failed: {e}. Tracking will be disabled.")
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score)
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

# ── Verify GPU ─────────────────────────────────────────────────────────────
log.info(f"TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
log.info(f"GPU devices: {gpus}")
if not gpus:
    raise RuntimeError(
        "❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU\n"
        "   Training will be extremely slow on CPU."
    )

# ── Global constants ────────────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 32
SEED         = 42
NUM_CLASSES  = 4
EPOCHS       = 15          # Phase 1 (feature extraction)
FT_EPOCHS    = 10          # Phase 2 (fine-tuning)

# Sorted alphabetically — MUST match flow_from_directory ordering
CLASSES = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']

# ── Per-model preprocessing map ────────────────────────────────────────────
# Each model expects different input preprocessing from ImageNet pretraining
PREPROCESS_MAP = {
    'EfficientNetB3': effnet_preprocess,   # scales to [-1, 1]
    'DenseNet121':    densenet_preprocess,  # caffe-style (BGR, mean subtraction)
    'ResNet50':       resnet_preprocess,    # caffe-style (BGR, mean subtraction)
}

log.info("✅ All imports successful. Constants set.")

## 🔍 Step 4 — Data Exploration

In [ ]:
# ── Count images per class ─────────────────────────────────────────────────
print("📊 Dataset Statistics")
print("=" * 50)
total = 0
counts = {}
for cls in CLASSES:
    img_dir = os.path.join(RAW_DATASET_PATH, cls, 'images')
    if not os.path.isdir(img_dir):
        img_dir = os.path.join(RAW_DATASET_PATH, cls)
    if not os.path.isdir(img_dir):
        log.error(f"❌ Class directory not found: {cls}")
        log.error(f"   Expected at: {os.path.join(RAW_DATASET_PATH, cls)}")
        raise FileNotFoundError(f"Missing class directory for '{cls}'")

    n = len([f for f in os.listdir(img_dir)
             if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    counts[cls] = n
    total += n
    print(f"  {cls:<22}: {n:>6,} images")
print("=" * 50)
print(f"  {'TOTAL':<22}: {total:>6,} images")

# ── Imbalance warning ──────────────────────────────────────────────────────
max_cls = max(counts, key=counts.get)
min_cls = min(counts, key=counts.get)
ratio = counts[max_cls] / counts[min_cls]
if ratio > 3.0:
    log.warning(f"⚠️  Class imbalance detected: '{max_cls}' has {ratio:.1f}x more "
                f"images than '{min_cls}'. Class weights will be used during training.")

# ── Charts ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
colors = ['#E74C3C', '#F39C12', '#27AE60', '#3498DB']

axes[0].bar(counts.keys(), counts.values(), color=colors,
            edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(axes[0].patches, counts.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

axes[1].pie(counts.values(), labels=counts.keys(), colors=colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# ── Sample images with None guard ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Sample X-ray from Each Class', fontsize=14, fontweight='bold')

for ax, cls, color in zip(axes, CLASSES, colors):
    img_dir = os.path.join(RAW_DATASET_PATH, cls, 'images')
    if not os.path.isdir(img_dir):
        img_dir = os.path.join(RAW_DATASET_PATH, cls)
    imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.png')])
    if not imgs:
        ax.text(0.5, 0.5, 'No images', ha='center', va='center')
        ax.set_title(cls)
        ax.axis('off')
        continue

    img = cv2.imread(os.path.join(img_dir, imgs[0]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        log.warning(f"cv2.imread returned None for {imgs[0]} in class {cls}")
        ax.text(0.5, 0.5, 'Read error', ha='center', va='center')
        ax.set_title(cls)
        ax.axis('off')
        continue

    ax.imshow(img, cmap='gray')
    ax.set_title(cls, fontweight='bold', fontsize=11,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=color,
                           alpha=0.7, edgecolor='none'),
                 color='white')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 🗂️ Step 5 — Prepare Dataset (Train / Val / Test Split)

Splits data **per class** into a flat directory structure on Google Drive.
**Persists across sessions** — only runs once, skips if already done.

| Split | Ratio | Purpose |
|-------|-------|---------|
| Train | 70%   | Model training |
| Val   | 15%   | Hyperparameter tuning & early stopping |
| Test  | 15%   | Final evaluation (never seen during training) |

In [ ]:


def split_is_valid(dest_dir, classes, min_per_class=50):
    '''Check if a prepared train/val/test split exists and has enough images.'''
    for split in ['train', 'val', 'test']:
        for cls in classes:
            cls_path = os.path.join(dest_dir, split, cls)
            if not os.path.isdir(cls_path):
                return False
            n = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            if n < min_per_class:
                return False
    return True


def prepare_dataset(source_dir, dest_dir, train_r=0.70):
    '''
    Split raw dataset into train/val/test on Google Drive.
    Skips entirely if a valid split already exists.
    '''
    if split_is_valid(dest_dir, CLASSES):
        log.info(f"✅ Dataset split already exists at {dest_dir}")
        # Print counts
        for split in ['train', 'val', 'test']:
            n = sum(len(os.listdir(os.path.join(dest_dir, split, c)))
                    for c in CLASSES)
            log.info(f"   {split:<6}: {n:,} images")
        return

    log.info("📁 Splitting dataset into train / val / test ...")
    log.info("-" * 60)

    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            os.makedirs(os.path.join(dest_dir, split, cls), exist_ok=True)

    for cls in CLASSES:
        img_dir = os.path.join(source_dir, cls, 'images')
        if not os.path.isdir(img_dir):
            img_dir = os.path.join(source_dir, cls)

        all_imgs = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

        if len(all_imgs) == 0:
            raise RuntimeError(f"❌ No images found for class '{cls}' in {img_dir}")

        train_imgs, temp = train_test_split(
            all_imgs, test_size=1 - train_r, random_state=SEED)
        val_imgs, test_imgs = train_test_split(
            temp, test_size=0.5, random_state=SEED)

        for split_name, split_imgs in [
            ('train', train_imgs), ('val', val_imgs), ('test', test_imgs)
        ]:
            for img_f in tqdm(split_imgs,
                              desc=f"Copying {cls} → {split_name}",
                              unit='img',
                              leave=False):
                src = os.path.join(img_dir, img_f)
                dst = os.path.join(dest_dir, split_name, cls, img_f)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

        log.info(f"  {cls:<22}: {len(train_imgs):>5} train | "
                 f"{len(val_imgs):>4} val | {len(test_imgs):>4} test")

    log.info("-" * 60)
    log.info("✅ Dataset split complete and saved to Drive!")


try:
    prepare_dataset(RAW_DATASET_PATH, DATASET_DIR)
except Exception as e:
    log.error(f"❌ Dataset preparation failed: {e}")
    raise

In [ ]:
import gc
import glob
import logging
import os
import shutil
import warnings

warnings.filterwarnings('ignore')
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator

drive.mount('/content/drive')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('PulmoVision_PostTraining')

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = 4
CLASS_NAMES = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
MODEL_NAMES = ['EfficientNetB3', 'DenseNet121']
PREPROCESS_MAP = {
    'EfficientNetB3': effnet_preprocess,
    'DenseNet121': densenet_preprocess,
}
CANDIDATE_ROOTS = [
    '/content/drive/MyDrive/Pulmoo',
    '/content/drive/MyDrive/Pulmooo',
    '/content/drive/MyDrive/Pulmmoo',
    '/content/drive/MyDrive/PulmoVision',
]


def find_dataset_dir():
    for root in CANDIDATE_ROOTS:
        candidate = os.path.join(root, 'dataset_split')
        if all(os.path.isdir(os.path.join(candidate, split)) for split in ['train', 'val', 'test']):
            return candidate
    matches = glob.glob('/content/drive/MyDrive/**/dataset_split/train', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    raise FileNotFoundError('dataset_split with train/val/test folders was not found in Drive.')


def find_model_path(model_name):
    filename = f'{model_name}_best.h5'
    for root in CANDIDATE_ROOTS:
        candidate = os.path.join(root, 'models', filename)
        if os.path.exists(candidate):
            return candidate
    matches = glob.glob(f'/content/drive/MyDrive/**/{filename}', recursive=True)
    return matches[0] if matches else None


def find_last_conv_layer(model_or_layer):
    for layer in reversed(model_or_layer.layers):
        if isinstance(layer, (
            tf.keras.layers.Conv2D,
            tf.keras.layers.SeparableConv2D,
            tf.keras.layers.DepthwiseConv2D,
        )):
            return layer
        if isinstance(layer, tf.keras.Model):
            nested_layer = find_last_conv_layer(layer)
            if nested_layer is not None:
                return nested_layer
    return None


DATASET_DIR = find_dataset_dir()
PROJECT_DIR = os.path.dirname(DATASET_DIR)
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
LOGS_DIR = os.path.join(PROJECT_DIR, 'logs')
os.makedirs(LOGS_DIR, exist_ok=True)

print('Using project dir:', PROJECT_DIR, flush=True)
print('Using dataset dir :', DATASET_DIR, flush=True)
print('Using models dir  :', MODELS_DIR, flush=True)


def create_generators(preprocess_fn, batch_size=BATCH_SIZE):
    train_dg = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        horizontal_flip=True,
        rotation_range=10,
        zoom_range=0.10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        fill_mode='nearest',
    )
    val_test_dg = ImageDataGenerator(preprocessing_function=preprocess_fn)

    train_gen = train_dg.flow_from_directory(
        os.path.join(DATASET_DIR, 'train'),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode='categorical',
        seed=SEED,
        shuffle=True,
    )
    val_gen = val_test_dg.flow_from_directory(
        os.path.join(DATASET_DIR, 'val'),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode='categorical',
        seed=SEED,
        shuffle=False,
    )
    test_gen = val_test_dg.flow_from_directory(
        os.path.join(DATASET_DIR, 'test'),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode='categorical',
        seed=SEED,
        shuffle=False,
    )
    return train_gen, val_gen, test_gen

## Step 13 — Load Saved Models

This post-training notebook loads only two model files:
- EfficientNetB3
- DenseNet121

ResNet50 is intentionally omitted from this variant.

In [ ]:
print('Step 13: loading saved models...', flush=True)

loaded_models = {}
model_paths = {}

for model_name in MODEL_NAMES:
    path = find_model_path(model_name)
    model_paths[model_name] = path
    print(f'{model_name} checkpoint: {path}', flush=True)
    if path and os.path.exists(path):
        try:
            loaded_models[model_name] = tf.keras.models.load_model(path)
            print(f'Loaded {model_name} from {path}', flush=True)
        except Exception as e:
            log.error(f'Failed to load {model_name}: {e}')
            loaded_models[model_name] = None
    else:
        log.warning(f'{model_name} checkpoint not found')
        loaded_models[model_name] = None

efficientnet_model = loaded_models.get('EfficientNetB3')
densenet_model = loaded_models.get('DenseNet121')

loaded_names = [name for name, model in loaded_models.items() if model is not None]
if not loaded_names:
    raise RuntimeError('No models were loaded. Check the Drive paths.')

print('Loaded models:', ', '.join(loaded_names), flush=True)

## Steps 14 to 16 — Evaluate, Ensemble, and Plot Confusion Matrices

This section evaluates only EfficientNetB3 and DenseNet121, then averages their predictions for the ensemble report.
ResNet50 is not part of this notebook variant.

In [ ]:
def evaluate_model(model, model_name):
    if model is None:
        print(f'Skipping {model_name}: model is not loaded.', flush=True)
        return None, None, None

    preprocess_fn = PREPROCESS_MAP[model_name]
    _, _, test_gen = create_generators(preprocess_fn)
    test_gen.reset()

    print(f'Evaluating {model_name} on {test_gen.samples} test images...', flush=True)
    proba = model.predict(test_gen, verbose=1)
    preds = np.argmax(proba, axis=1)
    truth = test_gen.classes
    acc = accuracy_score(truth, preds)

    print('\n' + '─' * 55, flush=True)
    print(f'  {model_name}', flush=True)
    print(f'  Test Accuracy: {acc:.4f}  ({acc * 100:.2f}%)', flush=True)
    if acc >= 0.85:
        print('  ✅ Meets target (≥85%)', flush=True)
    else:
        print('  ⚠️  Below target (≥85%)', flush=True)
    print('─' * 55, flush=True)
    print(classification_report(truth, preds, target_names=CLASS_NAMES), flush=True)

    return proba, preds, truth


eff_proba, eff_preds, eff_true = evaluate_model(efficientnet_model, 'EfficientNetB3')
den_proba, den_preds, den_true = evaluate_model(densenet_model, 'DenseNet121')

y_true = next((truth for truth in [eff_true, den_true] if truth is not None), None)
if y_true is None:
    raise RuntimeError('No valid ground-truth labels were returned from model evaluation.')

print('\n' + '=' * 40, flush=True)
print('  INDIVIDUAL MODEL SUMMARY', flush=True)
print('=' * 40, flush=True)
for name, preds in [('EfficientNetB3', eff_preds), ('DenseNet121', den_preds)]:
    if preds is not None:
        acc = accuracy_score(y_true, preds)
        bar = '█' * int(acc * 40)
        print(f'  {name:<16}: {acc:.4f}  |{bar}', flush=True)
    else:
        print(f'  {name:<16}: NOT EVALUATED', flush=True)

available_probas = []
available_names = []
for name, proba in [('EfficientNetB3', eff_proba), ('DenseNet121', den_proba)]:
    if proba is not None:
        available_probas.append(proba)
        available_names.append(name)

ensemble_proba = None
ensemble_preds = None
if len(available_probas) >= 2:
    ensemble_proba = np.mean(available_probas, axis=0)
    ensemble_preds = np.argmax(ensemble_proba, axis=1)
    ensemble_acc = accuracy_score(y_true, ensemble_preds)

    print('\n' + '=' * 55, flush=True)
    print('  ENSEMBLE SOFT VOTING — RESULTS', flush=True)
    print('=' * 55, flush=True)

    models_acc = {name: accuracy_score(y_true, preds) for name, preds in [('EfficientNetB3', eff_preds), ('DenseNet121', den_preds)] if preds is not None}
    models_acc['Ensemble'] = ensemble_acc

    for name, acc in models_acc.items():
        marker = ' ◀ BEST' if acc == max(models_acc.values()) else ''
        print(f'  {name:<18}: {acc:.4f}  ({acc * 100:.2f}%){marker}', flush=True)
    print('=' * 55, flush=True)
    print('\nEnsemble Classification Report:', flush=True)
    print(classification_report(y_true, ensemble_preds, target_names=CLASS_NAMES), flush=True)
else:
    print('\nSkipping ensemble: both model probability arrays are required.', flush=True)

all_results = []
for name, preds in [('EfficientNetB3', eff_preds), ('DenseNet121', den_preds)]:
    if preds is not None:
        all_results.append((name, preds))
if ensemble_preds is not None:
    all_results.append(('Ensemble', ensemble_preds))

n_plots = len(all_results)
if n_plots == 0:
    print('⚠️ No model results available for confusion matrices.', flush=True)
else:
    cols = min(n_plots, 2)
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(9 * cols, 7 * rows))
    fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.01)

    if n_plots == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    cmaps = ['Blues', 'Greens', 'Oranges']
    for i, (name, preds) in enumerate(all_results):
        ax = axes[i]
        cm = confusion_matrix(y_true, preds)
        acc = accuracy_score(y_true, preds)
        sns.heatmap(
            cm, annot=True, fmt='d', cmap=cmaps[i % len(cmaps)],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'},
        )
        ax.set_title(f'{name}\nAccuracy: {acc:.4f}', fontweight='bold', fontsize=12)
        ax.set_xlabel('Predicted Label', fontsize=10)
        ax.set_ylabel('True Label', fontsize=10)
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
        plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout(pad=3.0)
    save_path = os.path.join(LOGS_DIR, 'confusion_matrices_two_model.png')
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    log.info(f'✅ Saved: {save_path}')
    plt.show()

## Step 17 — Grad-CAM++ Explainability

This section uses the test set and the model-specific preprocessing path to render Grad-CAM++ overlays.
EfficientNetB3 stays the default visualization model in this two-model notebook.

In [ ]:
from PIL import Image
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

raw_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
raw_test_gen = raw_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED,
    shuffle=False,
)


def generate_heatmap_overlay(model, model_name, img_preprocessed, img_display, class_idx):
    try:
        cam_layer = find_last_conv_layer(model)
        if cam_layer is None:
            raise RuntimeError(f'Unable to find a convolutional layer for {model_name}.')

        cam_layer_name = cam_layer.name
        print(f'Using Grad-CAM penultimate layer for {model_name}: {cam_layer_name}', flush=True)
        gradcam = GradcamPlusPlus(model, model_modifier=ReplaceToLinear(), clone=True)
        cam = gradcam(
            CategoricalScore([class_idx]),
            img_preprocessed,
            penultimate_layer=cam_layer_name,
            seek_penultimate_conv_layer=False,
        )

        cam = np.array(cam[0], dtype=np.float32)
        cam = np.nan_to_num(cam, nan=0.0, posinf=0.0, neginf=0.0)
        cam_min = float(cam.min())
        cam_max = float(cam.max())
        if cam_max > cam_min:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)

        heatmap = np.uint8(255 * cam)
        colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
        colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)

        original = (img_display[0] * 255).astype(np.uint8)
        resized = cv2.resize(colored, (original.shape[1], original.shape[0]))
        overlay = cv2.addWeighted(original, 0.60, resized, 0.40, 0)
        return heatmap, overlay
    except Exception as e:
        log.error(f'Grad-CAM failed for {model_name}, class {class_idx}: {e}')
        blank = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        blank3 = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return blank, blank3


def visualize_gradcam(model, model_name, n_samples=6):
    if model is None:
        print(f'Skipping Grad-CAM for {model_name}: model is not loaded.', flush=True)
        return

    raw_test_gen.reset()
    total_batches = max(1, int(np.ceil(raw_test_gen.samples / raw_test_gen.batch_size)))
    batch_to_use = np.random.randint(0, total_batches)

    imgs, labels = None, None
    for _ in range(batch_to_use + 1):
        imgs, labels = next(raw_test_gen)

    n_samples = min(n_samples, len(imgs))
    preprocess_fn = PREPROCESS_MAP[model_name]
    raw_255_batch = imgs[:n_samples] * 255.0
    preprocessed_batch = preprocess_fn(raw_255_batch.copy())
    pred_batch = model.predict(preprocessed_batch, verbose=0)

    fig = plt.figure(figsize=(14, n_samples * 3.2))
    fig.suptitle(f'Grad-CAM++ Explainability — {model_name}', fontsize=14, fontweight='bold')
    col_titles = ['Original X-ray', 'Activation Heatmap', 'Overlay (40%)']

    for i in range(n_samples):
        img_display = imgs[i:i + 1]
        img_preprocessed = np.expand_dims(preprocessed_batch[i], axis=0)
        true_idx = int(np.argmax(labels[i]))
        pred_proba = pred_batch[i]
        pred_idx = int(np.argmax(pred_proba))
        conf = pred_proba[pred_idx] * 100
        correct = '✅' if pred_idx == true_idx else '❌'
        heatmap, overlay = generate_heatmap_overlay(model, model_name, img_preprocessed, img_display, pred_idx)

        for j, (img_data, cmap, xlabel) in enumerate([
            (img_display[0], None, f'True: {CLASS_NAMES[true_idx]}'),
            (heatmap, 'jet', f'Pred: {CLASS_NAMES[pred_idx]} {correct}'),
            (overlay, None, f'Confidence: {conf:.1f}%'),
        ]):
            ax = fig.add_subplot(n_samples, 3, i * 3 + j + 1)
            ax.imshow(img_data, cmap=cmap)
            ax.set_xlabel(xlabel, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(col_titles[j], fontweight='bold', fontsize=10)

    plt.tight_layout(pad=1.5)
    path = os.path.join(LOGS_DIR, f'{model_name}_gradcam.png')
    plt.savefig(path, dpi=100, bbox_inches='tight')
    log.info(f'✅ Saved: {path}')
    plt.show()


visualize_gradcam(efficientnet_model, 'EfficientNetB3', n_samples=6)
# visualize_gradcam(densenet_model, 'DenseNet121', n_samples=4)

## Step 18 — FastAPI Backend (Local)

The generated backend loads only EfficientNetB3 and DenseNet121.
It returns the raw activation-map contract for the frontend and is started locally at `http://localhost:8000`.

In [ ]:
%%writefile /content/pulmovision_api.py
"""PulmoVision - FastAPI Backend (two-model post-training variant)"""
import io
import logging
import os

import cv2
import numpy as np
import tensorflow as tf
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

log = logging.getLogger('PulmoVision_API')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

IMG_SIZE = 224
CAM_SIZE = 14
CLASS_NAMES = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
MAX_MB = 10
CANDIDATE_MODEL_DIRS = [
    '/content/drive/MyDrive/Pulmoo/models',
    '/content/drive/MyDrive/Pulmooo/models',
    '/content/drive/MyDrive/Pulmmoo/models',
    '/content/drive/MyDrive/PulmoVision/models',
    os.path.join(os.getcwd(), 'models'),
]
PREPROCESS_FNS = {
    'EfficientNetB3': effnet_preprocess,
    'DenseNet121': densenet_preprocess,
}


def find_model_path(model_name):
    filename = f'{model_name}_best.h5'
    for root in CANDIDATE_MODEL_DIRS:
        candidate = os.path.join(root, filename)
        if os.path.exists(candidate):
            return candidate
    return None


def find_last_conv_layer(model_or_layer):
    for layer in reversed(model_or_layer.layers):
        if isinstance(layer, (
            tf.keras.layers.Conv2D,
            tf.keras.layers.SeparableConv2D,
            tf.keras.layers.DepthwiseConv2D,
        )):
            return layer
        if isinstance(layer, tf.keras.Model):
            nested_layer = find_last_conv_layer(layer)
            if nested_layer is not None:
                return nested_layer
    return None


app = FastAPI(
    title='PulmoVision',
    description='Chest X-ray classification with Grad-CAM++ explainability',
    version='2.0',
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=False,
    allow_methods=['*'],
    allow_headers=['*'],
)

log.info('Loading models...')
models = {}
for name in ['EfficientNetB3', 'DenseNet121']:
    path = find_model_path(name)
    if path and os.path.exists(path):
        try:
            models[name] = tf.keras.models.load_model(path)
            log.info(f'  OK  {name} <- {path}')
        except Exception as e:
            log.error(f'  FAIL  {name}: {e}')
    else:
        log.warning(f'  MISSING  {name}')

if not models:
    raise RuntimeError('No models found. Run the training notebook first or verify the model paths.')

best_model_name = 'EfficientNetB3' if 'EfficientNetB3' in models else list(models.keys())[0]
best_model = models[best_model_name]
best_cam_layer = find_last_conv_layer(best_model)
if best_cam_layer is None:
    raise RuntimeError(f'Unable to find a convolutional layer for Grad-CAM in {best_model_name}.')
best_cam_layer_name = best_cam_layer.name
log.info(f'Grad-CAM model: {best_model_name}')
log.info(f'Grad-CAM penultimate layer: {best_cam_layer_name}')


def preprocess_for_model(image_bytes, model_name):
    try:
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    except Exception as e:
        raise HTTPException(400, detail=f'Cannot read image: {e}')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)
    fn = PREPROCESS_FNS.get(model_name)
    return fn(arr.copy()) if fn else arr


def make_activation_map(model, img_preprocessed, class_idx, size=CAM_SIZE):
    try:
        gradcam = GradcamPlusPlus(model, model_modifier=ReplaceToLinear(), clone=True)
        cam = gradcam(
            CategoricalScore([class_idx]),
            img_preprocessed,
            penultimate_layer=best_cam_layer_name,
            seek_penultimate_conv_layer=False,
        )
        activation_map = np.array(cam[0], dtype=np.float32)
        activation_map = np.nan_to_num(activation_map, nan=0.0, posinf=0.0, neginf=0.0)
        activation_map = np.clip(activation_map, 0.0, 1.0)
        activation_map = tf.image.resize(
            activation_map[..., np.newaxis], [size, size], method='bilinear'
        ).numpy()[..., 0]
        activation_map = np.clip(activation_map, 0.0, 1.0)
        return activation_map
    except Exception as e:
        log.error(f'Activation map generation failed: {e}')
        return np.zeros((size, size), dtype=np.float32)


@app.get('/health')
def health():
    return {
        'status': 'ok',
        'models_loaded': list(models.keys()),
        'cors': 'enabled',
    }


@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    allowed = {'image/jpeg', 'image/jpg', 'image/png'}
    if file.content_type not in allowed:
        raise HTTPException(400, detail='Only JPEG and PNG images are accepted.')

    contents = await file.read()
    if len(contents) > MAX_MB * 1024 * 1024:
        raise HTTPException(400, detail=f'File size exceeds {MAX_MB} MB limit.')

    probas = []
    for name, model in models.items():
        arr = preprocess_for_model(contents, name)
        probas.append(model.predict(arr, verbose=0)[0])

    avg = np.mean(probas, axis=0).astype(np.float32)
    avg = np.nan_to_num(avg, nan=0.0, posinf=0.0, neginf=0.0)
    avg = np.clip(avg, 0.0, None)
    total = float(np.sum(avg))
    if total <= 0.0:
        raise HTTPException(500, detail='Invalid probability sum from ensemble output.')
    avg = avg / total

    idx = int(np.argmax(avg))
    confidence_sum = float(np.sum(avg))
    confidence = {CLASS_NAMES[i]: round(float(avg[i]), 6) for i in range(len(CLASS_NAMES))}

    img_preprocessed = preprocess_for_model(contents, best_model_name)
    activation_map = make_activation_map(best_model, img_preprocessed, idx)

    return {
        'predicted_class': CLASS_NAMES[idx],
        'confidence': confidence,
        'confidence_sum': confidence_sum,
        'confidence_tolerance_ok': 0.9999 <= confidence_sum <= 1.0001,
        'activation_map': activation_map.tolist(),
        'activation_map_shape': [CAM_SIZE, CAM_SIZE],
        'activation_map_origin': 'top_left',
        'activation_map_encoding': 'normalized_float32',
    }

In [ ]:
import threading
import time

import requests
import uvicorn


def run_server():
    uvicorn.run('pulmovision_api:app', host='0.0.0.0', port=8000, log_level='info')


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
log.info('Server starting in background ...')
time.sleep(3)

api_url = 'http://localhost:8000'

print('=' * 65)
print('  PulmoVision API is LIVE!')
print('=' * 65)
print(f'  Local URL   : {api_url}')
print(f'  Health      : {api_url}/health')
print(f'  Predict     : POST {api_url}/predict')
print()
print('  Copy this URL into your frontend config as API_BASE_URL.')
print('=' * 65)

### Quick API Test

Run this after the backend starts to confirm the `/health` endpoint responds and the local server is live.

In [ ]:
for attempt in range(5):
    try:
        response = requests.get(f'{api_url}/health', timeout=5)
        if response.status_code == 200:
            print(f'✅ Health check passed on attempt {attempt + 1}:', flush=True)
            print(response.json(), flush=True)
            break
    except requests.exceptions.ConnectionError:
        if attempt < 4:
            log.info(f'Server not ready yet, retrying ({attempt + 1}/5) ...')
            time.sleep(2)
        else:
            log.error('❌ Server failed to respond after 5 attempts.')
            log.error('Check the server output above for errors.')

---

## Notes

- This notebook is the post-training companion for the original workflow.
- It only loads EfficientNetB3 and DenseNet121.
- ResNet50 is intentionally removed from the Step 13+ path.
- The backend keeps the raw activation-map response contract for the frontend.
- If you need to rerun earlier training steps, use the original [PulmoVision_Colab.ipynb](PulmoVision_Colab.ipynb).